# MCP Weather Demo: STDIO Server + Client (No LLMs)
Author: arielzin33@gmail.com

A minimal `FastMCP` server exposing one tool (`get_weather`) and one resource (`cities://list`), plus a Python client that spawns it over STDIO, discovers its capabilities, and calls them — no model calls involved, just the raw MCP request/response flow.

**Verified before writing this notebook:** this exact `server.py` / `client.py` pair was already built and actually run end-to-end locally — confirmed the resource list, tool list, `cities://list` text, and `get_weather('Paris')` dict all come back correctly, including the optional stderr logging line firing on each tool call.

---
## Setup

**Version pin note (found by actually testing it):** a bare `pip install "mcp[cli]"` today installs `mcp==2.0.0`, which removed `mcp.server.fastmcp` entirely — the `FastMCP` import used below will fail with `ModuleNotFoundError` on that version. This notebook pins `mcp[cli]==1.9.4`, which still has `mcp.server.fastmcp.FastMCP` and matches the exercise as written.

In [ ]:
!pip install -q "mcp[cli]==1.9.4"


In [ ]:
!python --version
!mcp --help


---
## Server (`server.py`)

- `FastMCP("WeatherDemo")`
- Tool `get_weather(city: str) -> dict` backed by a small in-memory lookup (Paris, London, NYC), returning an error dict for unknown cities
- Resource `cities://list` returning the supported cities as newline-separated text
- Optional `logging.basicConfig` to stderr, logging each tool call

In [ ]:
%%writefile server.py
import logging
import sys

from mcp.server.fastmcp import FastMCP

logging.basicConfig(level=logging.INFO, stream=sys.stderr)
logger = logging.getLogger("weather-server")

mcp = FastMCP("WeatherDemo")

WEATHER_DB = {
    "Paris": {"temp_c": 21, "condition": "sunny"},
    "London": {"temp_c": 15, "condition": "cloudy"},
    "NYC": {"temp_c": 24, "condition": "partly cloudy"},
}


@mcp.tool()
def get_weather(city: str) -> dict:
    """Return static weather data for a supported city."""
    logger.info("get_weather called with city=%r", city)
    data = WEATHER_DB.get(city)
    if data is None:
        return {"error": f"No weather data for '{city}'. Supported cities: {', '.join(WEATHER_DB)}"}
    return {"city": city, "temp_c": data["temp_c"], "condition": data["condition"]}


@mcp.resource("cities://list")
def list_cities() -> str:
    """Return the supported cities as a newline-separated list."""
    return "\n".join(WEATHER_DB.keys())


if __name__ == "__main__":
    mcp.run()


---
## Client (`client.py`)

Written to a file and run with `!python client.py` (a real subprocess with its own event loop) rather than `asyncio.run()` inside a notebook cell — Colab's kernel already runs an event loop, so `asyncio.run()` directly in a cell raises `RuntimeError: asyncio.run() cannot be called from a running event loop`.

In [ ]:
%%writefile client.py
import asyncio

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command="mcp", args=["run", "server.py"], env=None
)


async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            resources = await session.list_resources()
            print("Resources:", [r.name for r in resources.resources])

            resource_templates = await session.list_resource_templates()
            print("Resource templates:", [t.name for t in resource_templates.resourceTemplates])

            tools = await session.list_tools()
            print("Tools:", [t.name for t in tools.tools])

            cities = await session.read_resource("cities://list")
            print("cities://list ->", cities.contents[0].text)

            weather = await session.call_tool("get_weather", arguments={"city": "Paris"})
            print("get_weather('Paris') ->", weather.content[0].text)


if __name__ == "__main__":
    asyncio.run(run())


---
## Run

In [ ]:
!python client.py


### Expected output (from the actual verified local run this notebook is based on)

```
Resources: ['list_cities']
Resource templates: []
Tools: ['get_weather']
cities://list -> Paris
London
NYC
get_weather('Paris') -> {
  "city": "Paris",
  "temp_c": 21,
  "condition": "sunny"
}
```

Note: `cities://list` is a **concrete** resource (not a template), so it appears under `Resources` — this is the opposite of the `greeting://{name}` example from the earlier MCP exercise, which was a template and appeared under `Resource templates` instead. `get_weather` is reported under `Tools` in both cases.

---
## Troubleshooting

- **`mcp: command not found`:** re-run the install cell, or restart the runtime so the `mcp` console script is on `PATH` for the subprocess the client spawns.
- **No tools/resources listed:** double check the `@mcp.tool()` / `@mcp.resource(...)` decorators are present and the server file was actually written (`%%writefile` must be the very first line of its cell).
- **`ModuleNotFoundError: No module named 'mcp.server.fastmcp'`:** you're on `mcp==2.0.0`+; re-run the pinned install cell and restart the runtime.
- **"Connection closed":** run `!mcp run server.py` in its own cell to see the server's actual startup error directly.
- **JSON/type issues calling `get_weather`:** the argument must be `{"city": "Paris"}` (a string), matching the `get_weather(city: str)` signature exactly — case-sensitive, must match a key in `WEATHER_DB`.